**Sudeshna Das
(MBA Business Analytics)_ Gen AI Assignment**

1. INSTALL LIBRARIES

In [30]:
!pip install -U -q \
langchain \
langchain-community \
faiss-cpu \
sentence-transformers \
google-generativeai \
gradio \
pypdf \
tiktoken

2. CREATE BANK DOCUMENT FOLDER

In [31]:
import os

os.makedirs("bank_docs", exist_ok=True)

print("Folder created successfully")

Folder created successfully


3. CREATE DATASET

In [32]:
##DOCUMENT 1 — PERSONAL LOAN
loan_policy = """
PERSONAL LOAN POLICY

A personal loan is an unsecured loan.

Interest Rate:
10% to 15% annually.

Loan Amount:
₹50,000 to ₹20,00,000

Eligibility:
- Age above 21
- Salaried or self-employed
- Minimum salary ₹25,000

Repayment tenure:
1 to 5 years
"""

with open("bank_docs/personal_loan.txt", "w") as f:
    f.write(loan_policy)

print("Personal loan file created")

Personal loan file created


In [33]:
##DOCUMENT 2 — HOME LOAN
home_loan = """
HOME LOAN POLICY

Home loan interest rate:
8.5% annually.

Maximum loan:
₹1 crore

Tenure:
Up to 30 years

Eligibility:
- Good credit score
- Stable income
"""

with open("bank_docs/home_loan.txt", "w") as f:
    f.write(home_loan)

print("Home loan file created")

Home loan file created


In [34]:
##DOCUMENT 3 — CREDIT CARD
credit_card = """
CREDIT CARD POLICY

Gold Card:
Annual fee ₹499

Platinum Card:
Annual fee ₹999

Benefits:
- Cashback
- Lounge access
- Reward points

Interest:
3.5% monthly
"""

with open("bank_docs/credit_card.txt", "w") as f:
    f.write(credit_card)

print("Credit card file created")

Credit card file created


In [35]:
##DOCUMENT 4 — FAQ
faq = """
BANKING FAQ

Q: How to open savings account?
A: Submit Aadhaar and PAN.

Q: What is minimum balance?
A: ₹5,000.

Q: How to reset password?
A: Use forgot password option.
"""

with open("bank_docs/banking_faq.txt", "w") as f:
    f.write(faq)

print("FAQ file created")

FAQ file created


4. VERIFY FILES

In [36]:
os.listdir("bank_docs")

['home_loan.txt', 'credit_card.txt', 'personal_loan.txt', 'banking_faq.txt']

5. LOAD DOCUMENTS

In [37]:
from langchain_community.document_loaders import TextLoader
import os

documents = []

for file in os.listdir("bank_docs"):

    path = f"bank_docs/{file}"

    loader = TextLoader(path)

    docs = loader.load()

    documents.extend(docs)

print("Total documents loaded:", len(documents))

Total documents loaded: 4


6. CHECK DOCUMENT CONTENT

In [38]:
print(documents[0].page_content)


HOME LOAN POLICY

Home loan interest rate:
8.5% annually.

Maximum loan:
₹1 crore

Tenure:
Up to 30 years

Eligibility:
- Good credit score
- Stable income



7. TEXT CHUNKING

In [39]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50
)

chunks = splitter.split_documents(documents)

print("Total chunks:", len(chunks))

Total chunks: 4


8. CHECK CHUNKS

In [40]:
print(chunks[0].page_content)

HOME LOAN POLICY

Home loan interest rate:
8.5% annually.

Maximum loan:
₹1 crore

Tenure:
Up to 30 years

Eligibility:
- Good credit score
- Stable income


9. LOAD EMBEDDING MODEL

In [41]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer(
    'paraphrase-MiniLM-L3-v2'
)

print("Embedding model loaded")

Loading weights:   0%|          | 0/55 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/paraphrase-MiniLM-L3-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded


10. CREATE EMBEDDINGS

In [42]:
texts = [chunk.page_content for chunk in chunks]

embeddings = embedding_model.encode(texts)

print(embeddings.shape)

(4, 384)


11. CREATE FAISS VECTOR DATABASE

In [43]:
import faiss
import numpy as np

dimension = embeddings.shape[1]

index = faiss.IndexFlatL2(dimension)

index.add(np.array(embeddings))

print("FAISS index created")

FAISS index created


12. CREATE CHUNK MAPPING

In [44]:
chunk_mapping = {}

for i, chunk in enumerate(chunks):

    chunk_mapping[i] = chunk.page_content

print("Chunk mapping ready")

Chunk mapping ready


13. GEMINI API SETUP

In [45]:
!pip install -U google-generativeai

In [46]:
import google.generativeai as genai

GOOGLE_API_KEY = "API Key"

genai.configure(
    api_key=GOOGLE_API_KEY
)

model = genai.GenerativeModel(
    "gemini-2.5-flash"
)

response = model.generate_content(
    "Hello"
)

print(response.text)

ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2399.71ms


Hello! How can I help you today?


In [47]:
for m in genai.list_models():
  if 'generateContent' in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/antigravity-preview-05-2026
models/

14. CREATE RETRIEVAL FUNCTION

In [48]:
def retrieve_context(query, top_k=1):

    query_embedding = embedding_model.encode([query])

    distances, indices = index.search(
        np.array(query_embedding),
        top_k
    )

    retrieved_chunks = []

    for idx in indices[0]:

        retrieved_chunks.append(
            chunk_mapping[idx]
        )

    return "\n".join(retrieved_chunks)

15. TEST RETRIEVAL

In [49]:
print(
    retrieve_context(
        "What is home loan interest rate?"
    )
)

HOME LOAN POLICY

Home loan interest rate:
8.5% annually.

Maximum loan:
₹1 crore

Tenure:
Up to 30 years

Eligibility:
- Good credit score
- Stable income


16. CREATE CHAT MEMORY

In [50]:
chat_history = []

17. CREATE FINAL RAG CHATBOT

In [51]:
def ask_chatbot(query):

    context = retrieve_context(query)

    prompt = f"""
Context:
{context}

Question:
{query}

Answer shortly.
"""

    response = model.generate_content(prompt)

    return response.text

18. TEST CHATBOT

In [52]:
response = ask_chatbot(
    "What is personal loan interest rate?"
)

print(response)

10% to 15% annually.


19. TEST MEMORY

In [53]:
ask_chatbot("Tell me about Platinum card")

'The Platinum Card has an annual fee of ₹999. It offers benefits such as cashback, lounge access, and reward points, with an interest rate of 3.5% monthly.'

In [54]:
ask_chatbot("What is its annual fee?")

'The provided policy does not mention an annual fee.'

20. CREATE CHATBOT UI

In [55]:
import gradio as gr

def chatbot_interface(message, history):

    response = ask_chatbot(message)

    return response

demo = gr.ChatInterface(
    fn=chatbot_interface,
    title="Banking RAG Chatbot",
    description="AI Banking Assistant"
)

demo.launch(
    share=True,
    show_error=True
)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f76dc1a3f4bcb31c84.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


21. API ENDPOINTS

In [60]:
from fastapi import FastAPI, UploadFile, File
from pydantic import BaseModel

app = FastAPI()

class ChatRequest(BaseModel):
    query: str

@app.get("/")
def root():
    return {"message": "Banking Chatbot API"}

@app.get("/health")
def health():
    return {"status": "healthy"}

@app.post("/chat")
def chat(req: ChatRequest):

    answer = ask_chatbot(req.query)

    return {
        "query": req.query,
        "answer": answer
    }

@app.post("/upload")
async def upload(file: UploadFile = File(...)):

    return {
        "filename": file.filename,
        "message": "File uploaded successfully"
    }

In [59]:
def ask_chatbot(query):

    context = retrieve_context(query)

    prompt = f"""
Context:
{context}

Question:
{query}

Answer shortly.
"""

    response = model.generate_content(prompt)

    return response.text